# eval

> Evaluate any finished run and compare runs on one scale: the same IoU and object code, a threshold sweep, a held-out threshold choice, and one table.

Training tells you how a run did on its own metric. To decide between two models you need them **scored by the same code on the same patches**. This notebook is that code; the `segpt-eval` command (in `14_cli`) is a thin wrapper around `run_eval`.

The plan, in order:

1. **Per-patch scores** - IoU plus connected-component object counts for every class (`patch_scores`), summed over patches (`aggregate`), and object F1 (`object_f1`).
2. **Score a dataset at many thresholds** - run a model once, threshold the probabilities at each sweep point (`score_dataset`).
3. **Held-out threshold choice** - picking the best threshold on the same patches you report is optimistic. Split the *source images* in two halves, choose the threshold on one half, score the other, and do it both ways (`split_halves`, `held_out`).
4. **Seeds and the go/no-go rule** - runs that differ only by seed form an *arm*; an arm has a mean and a spread (`arm_summary`). `decide` says whether a candidate arm beats the best other arm by more than the noise.
5. **Load a finished run** - from `.hydra/config.yaml` and `checkpoints/best.pth` (an RF-DETR run: `checkpoints/last_ema.pth`), rebuild the val split and the model (`sweep_run`). An RF-DETR run is predicted once per patch at the lowest swept threshold, then cut at each threshold (`score_rfdetr`), so it lands in the same table as a U-Net run.
6. **One command** - `evaluate_run` for one run, `compare_runs` for the table and the fingerprint check, `run_eval` for the whole flow.

Classes come from the run's config (`data.class_names`), not from the code. **Threshold convention.** A U-Net pixel is foreground when `prob > t` (strict); an RF-DETR detection is kept when `confidence >= t` (inclusive). The two families also sweep different default grids, so a shared grid point such as 0.5 is not exactly comparable between them; compare held-out choices, not one fixed number. Object metrics are reported for every class; the held-out choice and the decision rule use one class, `objects_class`, which defaults to the **last** class (for chip/lifted data that is `lifted`, the rare one).

In [ ]:
#| default_exp eval

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

import numpy as np

from segmentation_tools_pt.metrics import object_metrics
from segmentation_tools_pt.rfdetr import RFDETR_THRESHOLDS, is_rfdetr_cfg, load_rfdetr_predictor


## 1. Per-patch scores

One patch gives a `(K, H, W)` boolean prediction and ground truth. For each class we want the IoU (two empty masks count as a perfect 1.0) and the object counts from `object_metrics`: how many ground-truth objects there are, how many were found, how many predicted blobs matched nothing.

Experiment on a tiny synthetic patch with two classes.

In [ ]:
import numpy as np
gt = np.zeros((2, 8, 8), bool); gt[0, :4, :4] = True; gt[1, 6, 6] = True
pr = np.zeros((2, 8, 8), bool); pr[0, :4, :3] = True; pr[1, 0, 7] = True       # class 1: object missed, one false alarm
print((pr[0] & gt[0]).sum(), (pr[0] | gt[0]).sum())                               # 12 / 16 -> IoU 0.75

In [ ]:
#| export
DEFAULT_THRESHOLDS = (0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 0.995, 0.999, 0.9995, 0.9999)


def iou(pred, gt):
    "IoU of two boolean masks; two empty masks score 1.0."
    union = (pred | gt).sum()
    return float((pred & gt).sum() / union) if union else 1.0


def patch_scores(pred, gt, classes):
    "One patch's row: IoU plus connected-component object counts (`object_metrics`) per class. `pred`, `gt` are `(K, H, W)` boolean."
    row = {}
    for k, name in enumerate(classes):
        o = object_metrics(pred[k].astype(np.uint8), gt[k].astype(np.uint8))
        row[f'iou_{name}'] = iou(pred[k], gt[k])
        row[f'{name}_total'] = o['total_gt']
        row[f'{name}_detected'] = o['detected']
        row[f'{name}_false_alarms'] = o['false_alarms']
    return row

In [ ]:
#| test
row = patch_scores(pr, gt, ('a', 'b'))
assert row['iou_a'] == 0.75 and row['a_total'] == 1 and row['a_detected'] == 1 and row['a_false_alarms'] == 0, row
assert row['iou_b'] == 0.0 and row['b_total'] == 1 and row['b_detected'] == 0 and row['b_false_alarms'] == 1, row
assert iou(np.zeros((4, 4), bool), np.zeros((4, 4), bool)) == 1.0

Summing over patches: IoU is the **mean of patch IoUs**; object counts are **summed** and recall is `detected / total`. Object F1 is `2*detected / (2*detected + missed + false alarms)` and is 1.0 when there is nothing to find or flag.

In [ ]:
#| export
def aggregate(rows, classes):
    "Mean IoU and summed object counts over patch rows, plus recall per class."
    rows = list(rows)
    out = {}
    for name in classes:
        total = sum(r[f'{name}_total'] for r in rows)
        detected = sum(r[f'{name}_detected'] for r in rows)
        out[f'iou_{name}'] = float(np.mean([r[f'iou_{name}'] for r in rows]))
        out[f'{name}_total'] = total
        out[f'{name}_detected'] = detected
        out[f'{name}_recall'] = detected / max(total, 1)
        out[f'{name}_false_alarms'] = sum(r[f'{name}_false_alarms'] for r in rows)
    return out


def object_f1(detected, total, false_alarms):
    "Object-level F1 = 2*detected / (2*detected + missed + false alarms); 1.0 when there is nothing to find or flag."
    denom = 2 * detected + (total - detected) + false_alarms
    return 2 * detected / denom if denom else 1.0


def ids_sha1(ids):
    "Fingerprint of a set of patch ids, to check two runs were scored on the same val patches."
    return hashlib.sha1('\n'.join(sorted(str(i) for i in ids)).encode()).hexdigest()

In [ ]:
#| test
a = aggregate([row, row], ('a', 'b'))
assert a['iou_a'] == 0.75 and a['a_total'] == 2 and a['b_recall'] == 0.0 and a['b_false_alarms'] == 2
assert object_f1(2, 2, 0) == 1.0 and object_f1(0, 0, 0) == 1.0 and abs(object_f1(1, 2, 0) - 2 / 3) < 1e-12
assert ids_sha1(['x', 'y']) == ids_sha1(['y', 'x']) != ids_sha1(['x', 'z'])
# three classes work the same way: nothing in the code knows chip/lifted
r3 = patch_scores(np.zeros((3, 4, 4), bool), np.zeros((3, 4, 4), bool), ('p', 'q', 'r'))
assert sorted(k for k in r3 if k.startswith('iou_')) == ['iou_p', 'iou_q', 'iou_r']

## 2. Score a dataset at many thresholds

The model runs **once** per batch; each sweep point just thresholds the same probabilities. `score_dataset` takes any model and any dataset whose items are dicts with `image`, `mask` and `sample_id` (what `SegmentationDataset` returns), so it can be tested with a stub. Optionally it keeps `(image, gt, pred)` for chosen patch ids for a review report.

Experiment: a stub "model" that returns the image as logits, on three synthetic patches.

In [ ]:
import torch
class _Stub(torch.nn.Module):
    def forward(self, x): return x
ds = [dict(image=torch.full((2, 8, 8), v), mask=torch.ones(2, 8, 8), sample_id=f'p{i}') for i, v in enumerate((-5., 0.2, 5.))]
m = _Stub()
print(torch.sigmoid(m(ds[1]['image']))[0, 0, 0])      # sigmoid(0.2)=0.55: above 0.5, below 0.9

In [ ]:
#| export
def score_dataset(model, ds, thresholds, classes, device='cpu', batch_size=16, report_ids=None, report_threshold=0.5, image_for=None):
    """Per-patch rows at every threshold: `({str(t): {sample_id: row}}, keep)`.

    A binary run (one class, masks of shape `(H, W)`) is scored like a one-class stack.

    `keep` maps `report_ids` to `(image, gt, pred)` (pred at `report_threshold`); `image_for(sample_id)` supplies the raw image.
    """
    import torch
    from torch.utils.data import DataLoader
    model.to(device).eval()
    rows = {str(t): {} for t in thresholds}
    keep = {}
    with torch.no_grad():
        for b in DataLoader(ds, batch_size=batch_size):
            prob = torch.sigmoid(model(b['image'].to(device))).cpu()
            gt = b['mask'].bool().numpy()
            if gt.ndim == 3: gt = gt[:, None]      # a binary run's mask has no class axis; `prob` is (B, 1, H, W)
            for k, sid in enumerate(b['sample_id']):
                if report_ids is not None and sid in report_ids:
                    keep[sid] = (image_for(sid) if image_for else b['image'][k].numpy(), gt[k], (prob[k] > report_threshold).numpy())
            for t in thresholds:
                pred = (prob > t).numpy()
                for k, sid in enumerate(b['sample_id']):
                    rows[str(t)][sid] = patch_scores(pred[k], gt[k], classes)
    return rows, keep

In [ ]:
#| test
rows, keep = score_dataset(_Stub(), ds, [0.5, 0.9], ('a', 'b'), report_ids={'p1'})
assert set(rows) == {'0.5', '0.9'} and set(rows['0.5']) == {'p0', 'p1', 'p2'}
assert rows['0.5']['p0']['iou_a'] == 0.0 and rows['0.5']['p2']['iou_a'] == 1.0
assert rows['0.5']['p1']['iou_a'] == 1.0 and rows['0.9']['p1']['iou_a'] == 0.0      # 0.55 passes 0.5, fails 0.9
assert set(keep) == {'p1'} and keep['p1'][2].all()                                   # pred kept at report_threshold 0.5

In [ ]:
#| test
# a binary run: masks are (H, W) with no class axis; the ground truth must not be read as "row k of the mask"
bds = [dict(image=torch.full((1, 8, 8), 5.), mask=torch.cat([torch.ones(4, 8), torch.zeros(4, 8)]), sample_id='b0')]
brows, bkeep = score_dataset(_Stub(), bds, [0.5], ('defect',), report_ids={'b0'})
r = brows['0.5']['b0']
assert r['defect_total'] == 1 and r['defect_detected'] == 1, r      # the top half is one object; the old code saw an empty first row
assert abs(r['iou_defect'] - 0.5) < 1e-9, r                         # all-on prediction vs top-half mask
assert bkeep['b0'][1].shape == (1, 8, 8)

### Scoring an RF-DETR run

`score_dataset` runs a network over batches and thresholds probabilities. RF-DETR is an instance model: one forward pass per patch gives detections with scores, and a sweep point is just "keep detections scoring at least `t`, paint them into a `(K, H, W)` map". So `score_rfdetr` predicts **once per patch at the lowest swept threshold** and rasterises per threshold; the rows it returns have the same shape as `score_dataset`'s, which is why everything downstream (held-out choice, the table, the fingerprint check) needs no RF-DETR special case.

Ground truth comes from the val dataset (same patches, same masks as for the U-Net runs); the image fed to RF-DETR is the raw patch from the cache (`image_for`), since the val transforms normalise for a U-Net and RF-DETR normalises itself.

In [ ]:
#| export
def score_rfdetr(predictor, ds, thresholds, classes, report_ids=None, report_threshold=0.5, image_for=None):
    """Per-patch rows at every threshold for an RF-DETR predictor: `({str(t): {sample_id: row}}, keep)`, like `score_dataset`.

    `predictor` is an `RFDETRPredictor`; `ds` items are dicts with `mask` and `sample_id`; `image_for(sample_id)` returns the raw patch.
    Each patch is predicted once, at the lowest of `thresholds` (and of `report_threshold` when `report_ids` is given).
    """
    floor = min(list(thresholds) + ([report_threshold] if report_ids else []))
    rows = {str(t): {} for t in thresholds}
    keep = {}
    for item in ds:
        sid, gt = item['sample_id'], item['mask'].bool().numpy()
        if gt.ndim == 2: gt = gt[None]      # binary run: mask without a class axis
        img = image_for(sid)
        if img.shape[:2] != gt.shape[1:]:
            raise ValueError(f'{sid}: image is {img.shape[:2]} but the mask is {gt.shape[1:]}; RF-DETR needs the cache patch size to equal train.crop_size')
        dets = predictor.detections(img, floor)
        for t in thresholds:
            rows[str(t)][sid] = patch_scores(predictor.stack(dets, gt.shape[1:], t) > 0, gt, classes)
        if report_ids is not None and sid in report_ids:
            keep[sid] = (img, gt, predictor.stack(dets, gt.shape[1:], report_threshold) > 0)
    return rows, keep


In [ ]:
#| test
from types import SimpleNamespace
from segmentation_tools_pt.rfdetr import RFDETRPredictor

class _FakeRFDETR:
    "Stands in for `RFDETR`: class 0 found at 0.9, class 1 at 0.4, plus a weak class 1 false alarm at 0.05."
    def __init__(self): self.calls = []
    def predict(self, img, threshold=0.5):
        self.calls.append(threshold)
        spec = [(0, 0.9, (slice(0, 4), slice(0, 4))), (1, 0.4, (slice(4, 8), slice(4, 8))), (1, 0.05, (slice(0, 2), slice(6, 8)))]
        spec = [s for s in spec if s[1] >= threshold]
        m = np.zeros((len(spec), 8, 8), bool)
        for k, (_, _, sl) in enumerate(spec): m[k][sl] = True
        return SimpleNamespace(mask=m, class_id=np.array([s[0] for s in spec], int), confidence=np.array([s[1] for s in spec]))

class _PatchSet(list):
    sample_ids = property(lambda self: [x['sample_id'] for x in self])

def _gt():
    g = torch.zeros(2, 8, 8); g[0, :4, :4] = 1; g[1, 4:, 4:] = 1; return g
pds = _PatchSet(dict(image=torch.zeros(1, 8, 8), mask=_gt(), sample_id=f'p{i}') for i in range(4))
fake = _FakeRFDETR()
rows, keep = score_rfdetr(RFDETRPredictor(fake, 2), pds, [0.03, 0.3, 0.5], ('chip', 'lifted'),
                          report_ids={'p1'}, report_threshold=0.3, image_for=lambda sid: np.zeros((8, 8), np.uint8))
assert fake.calls == [0.03] * 4                                                              # one prediction per patch, at the lowest threshold
assert set(rows) == {'0.03', '0.3', '0.5'} and set(rows['0.3']) == {'p0', 'p1', 'p2', 'p3'}
r = rows['0.3']['p0']
assert r['iou_chip'] == 1.0 and r['iou_lifted'] == 1.0 and r['lifted_detected'] == 1 and r['lifted_false_alarms'] == 0
assert rows['0.03']['p0']['lifted_false_alarms'] == 1 and rows['0.5']['p0']['lifted_detected'] == 0       # weak FA appears, 0.4 det drops
assert set(keep) == {'p1'} and keep['p1'][0].shape == (8, 8) and keep['p1'][2][1].sum() == 16               # pred at report_threshold 0.3


In [ ]:
#| test
# RF-DETR path: same shape rule for a binary run
class _P:
    def detections(self, img, thr): return None
    def stack(self, dets, size, t): m = np.zeros((1,) + tuple(size), np.float32); m[0, :4, :] = 1; return m
rrows, _ = score_rfdetr(_P(), [dict(image=None, mask=torch.cat([torch.ones(4, 8), torch.zeros(4, 8)]), sample_id='r0')], [0.5], ('defect',),
                        image_for=lambda sid: np.zeros((8, 8), np.uint8))
assert rrows['0.5']['r0']['defect_total'] == 1 and rrows['0.5']['r0']['iou_defect'] == 1.0

## 3. Held-out threshold choice

Choosing the best threshold on the patches you then report is optimistic. Instead: split the *source images* (a source image gives many neighbouring patches that must not leak across the split) into two disjoint halves by sorting their sha1 and alternating, pick the threshold on one half, score the other, and do it both ways. The pick is the threshold with the best object F1 on `objects_class`; ties go to the higher threshold.

The constants below are the ones fixed in advance for the #106 go/no-go rule; the numbers must not drift.

In [ ]:
#| export
MIN_GAIN = 0.01        # F1 gain a candidate needs over the best other arm, on top of the seed noise
NOISE_FACTOR = 2       # ... which must also exceed this many seed spreads
IOU_MARGIN = 0.01      # the candidate's IoU on `iou_class` may not trail the best other arm by more than this


def split_halves(source_ids):
    "Two disjoint, deterministic halves of the source images: sort by sha1, alternate."
    ordered = sorted(set(source_ids), key=lambda s: hashlib.sha1(str(s).encode()).hexdigest())
    return set(ordered[0::2]), set(ordered[1::2])


def held_out(by_threshold, stem_source, halves, classes, objects_class=None):
    """Select the threshold on one half (best `objects_class` object F1, ties to the higher threshold), score the other, both ways.

    `by_threshold` is `{threshold: {patch_id: row}}`; `stem_source` maps a patch id to its source image id. `objects_class`
    defaults to the last class. Returns one dict per direction with the aggregate, `f1_<objects_class>`, `threshold`, `selection_f1`.
    """
    cls = objects_class or classes[-1]
    out = []
    for sel, ev in ((halves[0], halves[1]), (halves[1], halves[0])):
        best_t, best_f1 = None, -1.0
        for t in sorted(by_threshold, key=float):
            a = aggregate([r for s, r in by_threshold[t].items() if stem_source[s] in sel], classes)
            f1 = object_f1(a[f'{cls}_detected'], a[f'{cls}_total'], a[f'{cls}_false_alarms'])
            if f1 >= best_f1:
                best_t, best_f1 = t, f1
        a = aggregate([r for s, r in by_threshold[best_t].items() if stem_source[s] in ev], classes)
        a[f'f1_{cls}'] = object_f1(a[f'{cls}_detected'], a[f'{cls}_total'], a[f'{cls}_false_alarms'])
        a['threshold'] = float(best_t); a['selection_f1'] = best_f1
        out.append(a)
    return out


def run_summary(directions):
    "Mean of the two directions for one run."
    keys = [k for k in directions[0] if k != 'threshold']
    s = {k: float(np.mean([d[k] for d in directions])) for k in keys}
    s['thresholds'] = [d['threshold'] for d in directions]
    return s

In [ ]:
#| test
ids = [f'src{i}' for i in range(10)]
ha, hb = split_halves(ids)
assert ha.isdisjoint(hb) and ha | hb == set(ids) and abs(len(ha) - len(hb)) <= 1
assert split_halves(ids) == (ha, hb) and split_halves(reversed(ids)) == (ha, hb)
stem_source = {f'{s}_{k}': s for s in ids for k in 'ab'}
CL = ('chip', 'lifted')

def _row(det, tot, fa):
    return dict(iou_chip=0.9, iou_lifted=0.8, chip_total=1, chip_detected=1, chip_false_alarms=0,
                lifted_total=tot, lifted_detected=det, lifted_false_alarms=fa)

by_t = {'0.1': {s: _row(2, 2, 3) for s in stem_source}, '0.5': {s: _row(1, 2, 0) for s in stem_source}}
d = held_out(by_t, stem_source, (ha, hb), CL)
assert [x['threshold'] for x in d] == [0.5, 0.5] and abs(d[0]['f1_lifted'] - 2 / 3) < 1e-9, d
assert d[0]['lifted_total'] == 2 * len([s for s in stem_source if stem_source[s] in hb]) and d[1]['lifted_false_alarms'] == 0
tie = {'0.1': {s: _row(1, 2, 0) for s in stem_source}, '0.5': {s: _row(1, 2, 0) for s in stem_source}}
assert [x['threshold'] for x in held_out(tie, stem_source, (ha, hb), CL)] == [0.5, 0.5]          # ties go to the higher threshold
# the pick uses only the selection half: 0.1 is great on half A, awful on half B
by_t2 = {'0.1': {s: _row(2, 2, 0) if stem_source[s] in ha else _row(0, 2, 9) for s in stem_source},
         '0.5': {s: _row(1, 2, 0) for s in stem_source}}
d2 = held_out(by_t2, stem_source, (ha, hb), CL)
assert d2[0]['threshold'] == 0.1 and d2[1]['threshold'] == 0.5       # A->B picks 0.1 on A; B->A picks 0.5 on B
assert d2[0]['lifted_detected'] == 0                                  # ... and then scores 0.1 on B: it does not transfer
# class names are free: the object class can be the first class too
d3 = held_out(by_t, stem_source, (ha, hb), CL, objects_class='chip')
assert 'f1_chip' in d3[0] and 'f1_lifted' not in d3[0]

## 4. Arms, seeds and the decision rule

Runs that differ only by seed form an **arm**. An arm's score is the mean over its runs; its spread is half the range for two runs, the standard deviation for three or more, 0 for one. `decide(arms, candidate, objects_class, iou_class)` takes both class names from the caller (a run's `objects_class` is in its `evaluate_run` result) and applies the rule written down before the #106 runs finished: the candidate arm goes only if its object F1 beats the best other arm by more than `max(MIN_GAIN, NOISE_FACTOR * larger spread)` **and** its `iou_class` IoU is not worse than the best arm's by more than `IOU_MARGIN`.

In [ ]:
#| export
def arm_summary(runs, objects_class=None, keys=None):
    "Mean over an arm's runs and the seed spread: half the range for 2 runs, the standard deviation for 3 or more, 0 for 1."
    if keys is None:
        cls = objects_class or [k[3:] for k in runs[0] if k.startswith('f1_')][-1]
        keys = [f'f1_{cls}', f'{cls}_recall', f'{cls}_false_alarms'] + [k for k in runs[0] if k.startswith('iou_')]
    out = {'n_runs': len(runs)}
    for k in keys:
        v = np.array([r[k] for r in runs], float)
        out[k] = float(v.mean())
        out[k + '_spread'] = 0.0 if len(v) < 2 else float((v.max() - v.min()) / 2 if len(v) == 2 else v.std())
    return out


def decide(arms, candidate, objects_class, iou_class):
    "Apply the rule: go only if the candidate's F1 on `objects_class` beats the best other arm by more than the noise and its `iou_class` IoU is not worse."
    f1, io = f'f1_{objects_class}', f'iou_{iou_class}'
    others = {n: a for n, a in arms.items() if n != candidate}
    best = max(others, key=lambda n: others[n][f1])
    c, b = arms[candidate], others[best]
    gain = c[f1] - b[f1]
    need = max(MIN_GAIN, NOISE_FACTOR * max(c[f1 + '_spread'], b[f1 + '_spread']))
    iou_ok = c[io] >= b[io] - IOU_MARGIN
    return {'best_semantic_arm': best, 'f1_gain': gain, 'required_gain': need, 'iou_ok': bool(iou_ok),
            'go': bool(gain > need and iou_ok)}

In [ ]:
#| test
_z = dict(lifted_recall=0, lifted_false_alarms=0, iou_lifted=0, iou_chip=0)
assert abs(arm_summary([dict(_z, f1_lifted=.5), dict(_z, f1_lifted=.6)])['f1_lifted_spread'] - .05) < 1e-9
assert arm_summary([dict(_z, f1_lifted=.5)])['f1_lifted_spread'] == 0.0
s3 = arm_summary([dict(_z, f1_lifted=v) for v in (.5, .6, .7)])
assert abs(s3['f1_lifted_spread'] - np.std([.5, .6, .7])) < 1e-12 and s3['n_runs'] == 3

def arm(f1, spread, chip=.9):
    return dict(f1_lifted=f1, f1_lifted_spread=spread, iou_chip=chip)
assert decide({'rf': arm(.80, .005), 'u': arm(.70, .005)}, 'rf', 'lifted', 'chip')['go']
assert not decide({'rf': arm(.705, .001), 'u': arm(.70, .001)}, 'rf', 'lifted', 'chip')['go']            # gain below MIN_GAIN
assert not decide({'rf': arm(.74, .03), 'u': arm(.70, .02)}, 'rf', 'lifted', 'chip')['go']               # gain below 2 * spread
assert not decide({'rf': arm(.80, .005, chip=.85), 'u': arm(.70, .005)}, 'rf', 'lifted', 'chip')['go']   # chip IoU worse by > 0.01
assert decide({'rf': arm(.80, .0), 'u': arm(.70, .0), 'u2': arm(.79, .0)}, 'rf', 'lifted', 'chip')['best_semantic_arm'] == 'u2'
assert set(decide({'rf': arm(.80, .0), 'u': arm(.70, .0)}, 'rf', 'lifted', 'chip')) == {'best_semantic_arm', 'f1_gain', 'required_gain', 'iou_ok', 'go'}
arm_ab = lambda f1, spread, io=.9: {'f1_b': f1, 'f1_b_spread': spread, 'iou_a': io}              # class names come from the caller
assert decide({'x': arm_ab(.8, 0), 'y': arm_ab(.7, 0)}, 'x', 'b', 'a')['go']
try:
    decide({'x': arm_ab(.8, 0), 'y': arm_ab(.7, 0)}, 'x'); raise AssertionError('classes must be required')
except TypeError: pass

## 5. Loading a finished run

A run directory holds `.hydra/config.yaml` and `checkpoints/best.pth` (a dict with `model_state`); an RF-DETR run (`model.source: rfdetr`) holds `checkpoints/last_ema.pth` instead and is dispatched to `score_rfdetr`. Everything the evaluation needs is in that config: the data root and mode, the class names, the model arch and source. We rebuild the val split with the same patch pipeline (it reuses the cache, nothing is re-extracted when it exists) and the same val transforms, load the weights, and call `score_dataset`.

Only patch mode is supported: held-out halves need the `source_id` of each patch from the cache's `metadata.json`.

In [ ]:
#| export
def _load_run(run_dir):
    "Config, patch root and val dataset of a finished run."
    from omegaconf import OmegaConf
    from segmentation_tools_pt.dataset import SegmentationDataset
    from segmentation_tools_pt.hydra_pipeline import resolve_channel_preset, resolve_mask_cfg, run_patch_pipeline
    from segmentation_tools_pt.transforms import get_val_transforms
    cfg_path = Path(run_dir) / '.hydra' / 'config.yaml'
    if not cfg_path.exists():
        raise FileNotFoundError(f'{cfg_path} not found: {run_dir} is not a finished segpt-train run')
    cfg = OmegaConf.load(cfg_path)
    if not cfg.data.get('patch_mode', True):
        raise ValueError(f'{run_dir}: segpt-eval needs a patch-mode run (data.patch_mode=true)')
    preset = resolve_channel_preset(cfg.data.data_mode)
    mask_cfg = resolve_mask_cfg(cfg)
    mean, std, root = run_patch_pipeline(cfg, Path(cfg.data.raw_dir), Path(cfg.data.patch_dir), preset['n_source_channels'],
                                         preset['channel_names'], cfg.data.mask_suffix or '')
    ds = SegmentationDataset(root, 'val', transform=get_val_transforms(cfg.train.crop_size, tuple(mean), tuple(std)),
                             num_classes=cfg.train.num_classes, n_source_channels=preset['n_source_channels'],
                             channel_names=preset['channel_names'], **mask_cfg)
    return cfg, preset, Path(root), ds, list(mask_cfg['class_names'])


def sweep_run(run_dir, thresholds=None, report_ids=None, report_threshold=0.5, device=None, batch_size=16, loader=None):
    """Score a finished run on its val split at every threshold: `best.pth` for a U-Net run, `last_ema.pth` for an RF-DETR run.

    `thresholds=None` means the run's own default grid: `DEFAULT_THRESHOLDS`, or `RFDETR_THRESHOLDS` for RF-DETR (its scores are
    not probabilities, the useful range sits lower). `loader(ckpt)` replaces the RF-DETR model loader (tests).

    Returns `(by_threshold, keep, val_ids_sha1, per_patch, info)` where `by_threshold` is `{str(t): aggregate}`, `per_patch`
    is `{str(t): {patch_id: row}}` and `info` carries `classes`, the patch `root` and the run `cfg`.
    """
    import cv2, torch
    from segmentation_tools_pt.hydra_pipeline import resolve_crop_size
    from segmentation_tools_pt.model import get_model_from_cfg
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    cfg, preset, root, ds, classes = _load_run(run_dir)
    image_for = lambda sid: cv2.imread(str(root / 'val' / 'images' / f'{sid}.png'), cv2.IMREAD_UNCHANGED)
    if is_rfdetr_cfg(cfg):
        thresholds = RFDETR_THRESHOLDS if thresholds is None else thresholds
        predictor = load_rfdetr_predictor(run_dir, num_classes=len(classes), loader=loader)
        rows, keep = score_rfdetr(predictor, ds, thresholds, classes, report_ids, report_threshold, image_for)
        by_t = {t: aggregate(r.values(), classes) for t, r in rows.items()}
        return by_t, keep, ids_sha1(ds.sample_ids), rows, {'classes': classes, 'root': root, 'cfg': cfg}
    thresholds = DEFAULT_THRESHOLDS if thresholds is None else thresholds
    model = get_model_from_cfg(cfg, preset['in_channels'], crop_size=resolve_crop_size(cfg), pretrained=False)
    ckpt = torch.load(Path(run_dir) / 'checkpoints' / 'best.pth', map_location='cpu', weights_only=True)
    model.load_state_dict(ckpt['model_state'])
    rows, keep = score_dataset(model, ds, thresholds, classes, device, batch_size, report_ids, report_threshold, image_for)
    by_t = {t: aggregate(r.values(), classes) for t, r in rows.items()}
    return by_t, keep, ids_sha1(ds.sample_ids), rows, {'classes': classes, 'root': root, 'cfg': cfg}

## 6. One run, many runs, one table

`evaluate_run` is the sweep plus, on request, the held-out choice. A run's **arm** is its directory name without a trailing `_s<seed>`, so `resnet18_e15_s42` and `resnet18_e15_s43` are two seeds of arm `resnet18_e15`. `compare_runs` groups results by arm, **refuses to compare runs whose val ids differ** (`check_fingerprints`), and builds one row per arm. With `--heldout` the row is the held-out mean over seeds; without it, the row is each run at one fixed threshold.

In [ ]:
#| export
def arm_name(run_dir):
    "A run's arm: its directory name without a trailing `_s<seed>`."
    return re.sub(r'_s\d+$', '', Path(run_dir).resolve().name)


def evaluate_run(run_dir, thresholds=None, heldout=False, objects_class=None, report_ids=None,
                 report_threshold=0.5, device=None, batch_size=16, loader=None):
    """Sweep one finished run; with `heldout=True` also choose the threshold on one half of the source images and score the other.

    Dispatches on the run config: an RF-DETR run is scored with `score_rfdetr`. Returns a JSON-ready dict; the heavy parts sit under `_per_patch` and `_keep` (drop them before dumping).
    """
    by_t, keep, sha, per_patch, info = sweep_run(run_dir, thresholds, report_ids, report_threshold, device, batch_size, loader)
    classes = info['classes']
    cls = objects_class or classes[-1]
    if cls not in classes:
        raise ValueError(f'objects_class {cls!r} is not one of the run classes {classes}')
    res = {'run': Path(run_dir).resolve().name, 'arm': arm_name(run_dir), 'run_dir': str(run_dir), 'classes': classes,
           'objects_class': cls, 'val_ids_sha1': sha, 'thresholds': by_t, '_per_patch': per_patch, '_keep': keep}
    if heldout:
        meta_path = info['root'] / 'metadata.json'
        stem_source = {e['sample_id']: e['source_id'] for e in json.loads(meta_path.read_text())['val']}
        if set(stem_source) != set(per_patch[next(iter(per_patch))]):
            raise ValueError(f'{meta_path}: val patch ids differ from the scored val split')
        halves = split_halves(stem_source.values())
        res['n_sources'] = [len(halves[0]), len(halves[1])]
        res['heldout'] = held_out(per_patch, stem_source, halves, classes, cls)
        res['heldout_summary'] = run_summary(res['heldout'])
    return res


def check_fingerprints(results):
    "Raise if the runs were not scored on the same val patches: a table of such runs would compare different things."
    shas = {r['run']: r['val_ids_sha1'] for r in results}
    if len(set(shas.values())) > 1:
        lines = '\n'.join(f'  {n}: {s}' for n, s in shas.items())
        raise ValueError(f'val-id fingerprints differ, refusing to compare:\n{lines}')
    classes = {tuple(r['classes']) for r in results}
    if len(classes) > 1:
        raise ValueError(f'class names differ between runs, refusing to compare: {sorted(classes)}')


def compare_runs(results, threshold=0.5):
    """Comparison dict for finished `evaluate_run` results: one row per arm, plus the fingerprint and class list.

    Rows come from the held-out summary when every run has one (mean over seeds, with spreads); otherwise from the sweep at `threshold`.
    """
    check_fingerprints(results)
    classes, cls = results[0]['classes'], results[0]['objects_class']
    if any(r['objects_class'] != cls for r in results):
        raise ValueError('runs were evaluated with different objects_class values')
    held = all('heldout_summary' in r for r in results)
    arms = {}
    for name in dict.fromkeys(r['arm'] for r in results):
        mine = [r for r in results if r['arm'] == name]
        if held:
            a = arm_summary([r['heldout_summary'] for r in mine], cls)
            a['thresholds'] = [r['heldout_summary']['thresholds'] for r in mine]
        else:
            t = str(threshold)
            if any(t not in r['thresholds'] for r in mine):
                raise ValueError(f'threshold {threshold} was not swept; swept: {sorted(mine[0]["thresholds"], key=float)}')
            runs = []
            for r in mine:
                a = r['thresholds'][t]
                runs.append(dict(a, **{f'f1_{cls}': object_f1(a[f'{cls}_detected'], a[f'{cls}_total'], a[f'{cls}_false_alarms'])}))
            a = arm_summary(runs, cls)
            a['thresholds'] = [float(threshold)]
        a['runs'] = [r['run'] for r in mine]
        arms[name] = a
    return {'val_ids_sha1': results[0]['val_ids_sha1'], 'classes': classes, 'objects_class': cls,
            'mode': 'heldout' if held else f'threshold={threshold}', 'arms': arms}


def format_table(cmp):
    "Plain-text table of a `compare_runs` dict: F1, recall, false alarms of the objects class and every class's IoU."
    cls, classes = cmp['objects_class'], cmp['classes']
    head = ['arm', 'n', f'F1_{cls}', f'recall_{cls}', f'FA_{cls}'] + [f'IoU_{c}' for c in classes] + ['threshold']
    lines = [head]
    for name, a in cmp['arms'].items():
        pm = lambda k, f='{:.3f}': f.format(a[k]) + (f' +-{a[k + "_spread"]:.3f}' if a['n_runs'] > 1 else '')
        th = sorted({t for ts in a['thresholds'] for t in (ts if isinstance(ts, list) else [ts])})
        lines.append([name, str(a['n_runs']), pm(f'f1_{cls}'), pm(f'{cls}_recall'), pm(f'{cls}_false_alarms', '{:.1f}')]
                     + [pm(f'iou_{c}') for c in classes] + [','.join(f'{t:g}' for t in th)])
    w = [max(len(r[i]) for r in lines) for i in range(len(head))]
    out = ['  '.join(c.ljust(w[i]) for i, c in enumerate(r)) for r in lines]
    out.insert(1, '  '.join('-' * x for x in w))
    return f"val ids {cmp['val_ids_sha1'][:12]} | {cmp['mode']} | classes {','.join(classes)}\n" + '\n'.join(out)

In [ ]:
#| test
def _res(run, sha='aaa', f1pairs=((2, 2, 0), (1, 2, 1)), held=False):
    th = {}
    for t, (det, tot, fa) in zip(('0.3', '0.5'), f1pairs):
        th[t] = dict(iou_chip=.9, iou_lifted=.8, chip_total=1, chip_detected=1, chip_false_alarms=0, chip_recall=1.0,
                     lifted_total=tot, lifted_detected=det, lifted_false_alarms=fa, lifted_recall=det / tot)
    r = dict(run=run, arm=arm_name('/x/' + run), val_ids_sha1=sha, classes=['chip', 'lifted'], objects_class='lifted', thresholds=th)
    if held:
        d = dict(th['0.5'], f1_lifted=0.5, threshold=0.5, selection_f1=.5)
        r['heldout'] = [d, d]; r['heldout_summary'] = run_summary([d, d])
    return r

assert arm_name('/a/b/resnet18_e15_s42') == arm_name('c/resnet18_e15_s7') == 'resnet18_e15' and arm_name('plain') == 'plain'
r1, r2 = _res('m1_s1'), _res('m2_s1', sha='bbb')
try:
    compare_runs([r1, r2]); raise AssertionError('fingerprint mismatch must raise')
except ValueError as e:
    assert 'fingerprints differ' in str(e) and 'm1_s1' in str(e) and 'm2_s1' in str(e)
try:
    compare_runs([r1, dict(_res('m3_s1'), classes=['a', 'b'])]); raise AssertionError('class mismatch must raise')
except ValueError as e:
    assert 'class names differ' in str(e)
cmp = compare_runs([r1, _res('m1_s2'), _res('m2_s1')])
assert list(cmp['arms']) == ['m1', 'm2'] and cmp['arms']['m1']['n_runs'] == 2 and abs(cmp['arms']['m1']['f1_lifted'] - 0.5) < 1e-9
tab = format_table(cmp)
assert 'F1_lifted' in tab and 'm1' in tab and 'm2' in tab and 'aaa' in tab, tab
try:
    compare_runs([r1], threshold=0.7); raise AssertionError('unswept threshold must raise')
except ValueError as e:
    assert 'not swept' in str(e)
cmph = compare_runs([_res('m1_s1', held=True), _res('m1_s2', held=True)])
assert cmph['mode'] == 'heldout' and cmph['arms']['m1']['f1_lifted'] == 0.5 and cmph['arms']['m1']['thresholds'] == [[0.5, 0.5], [0.5, 0.5]]
print(tab)

### An RF-DETR run in the same table

Run directory with `model.source: rfdetr`, a stub model instead of `rfdetr`, and the run loader replaced by a tiny in-memory val split. The question: does an RF-DETR run go through `sweep_run`, `evaluate_run` (sweep and held-out) and `compare_runs` next to a U-Net result, with its **own** default threshold grid?

In [ ]:
#| test
import cv2, tempfile
from omegaconf import OmegaConf

def _rf_run(root_dir, sha_ids=('p0', 'p1', 'p2', 'p3')):
    "A fake RF-DETR run dir plus the patch cache root and val dataset `_load_run` would return."
    run = Path(root_dir) / 'rf_small_s1'
    (run / 'checkpoints').mkdir(parents=True); (run / 'checkpoints' / 'last_ema.pth').write_bytes(b'x')
    cache = Path(root_dir) / 'cache'; (cache / 'val' / 'images').mkdir(parents=True)
    for s in sha_ids: cv2.imwrite(str(cache / 'val' / 'images' / f'{s}.png'), np.zeros((8, 8), np.uint8))
    (cache / 'metadata.json').write_text(json.dumps({'val': [dict(sample_id=s, source_id=f'src{k // 2}') for k, s in enumerate(sha_ids)]}))
    ds = _PatchSet(dict(image=torch.zeros(1, 8, 8), mask=_gt(), sample_id=s) for s in sha_ids)
    cfg = OmegaConf.create({'model': {'source': 'rfdetr', 'arch': 'small'}, 'data': {'class_names': ['chip', 'lifted']}})
    return run, (cfg, {}, cache, ds, ['chip', 'lifted'])

_orig_load_run = _load_run
try:
    with tempfile.TemporaryDirectory() as td:
        run, loaded = _rf_run(td)
        _load_run = lambda rd: loaded
        by_t, keep, sha, per_patch, info = sweep_run(run, loader=lambda ckpt: _FakeRFDETR())
        assert sorted(by_t, key=float) == [str(t) for t in RFDETR_THRESHOLDS], sorted(by_t)         # default grid is the RF-DETR one
        assert sha == ids_sha1(['p0', 'p1', 'p2', 'p3']) and info['classes'] == ['chip', 'lifted']
        by_t2, *_ = sweep_run(run, thresholds=[0.3, 0.5], loader=lambda ckpt: _FakeRFDETR())
        assert sorted(by_t2) == ['0.3', '0.5']                                                      # an explicit grid wins
        res = evaluate_run(run, heldout=True, loader=lambda ckpt: _FakeRFDETR())
        assert res['arm'] == 'rf_small' and res['objects_class'] == 'lifted' and len(res['heldout']) == 2
        assert [h['threshold'] for h in res['heldout']] == [0.3, 0.3] and res['heldout'][0]['f1_lifted'] == 1.0
        for k in ('_per_patch', '_keep'): res.pop(k)
        unet = _res('unet_s1', sha=res['val_ids_sha1'], held=True)                                  # a U-Net-style result, same val ids
        cmp_ = compare_runs([res, unet])
        assert list(cmp_['arms']) == ['rf_small', 'unet'] and cmp_['mode'] == 'heldout'
        assert cmp_['arms']['rf_small']['f1_lifted'] == 1.0 and cmp_['arms']['unet']['f1_lifted'] == 0.5
        assert 'rf_small' in format_table(cmp_) and 'unet' in format_table(cmp_)
        _, loaded_other = _rf_run(td + '/other', sha_ids=('p0', 'p1', 'p9'))                        # different val patches
        _load_run = lambda rd: loaded_other
        other = dict(evaluate_run(run, loader=lambda ckpt: _FakeRFDETR()), run='other_s1')
        try:
            compare_runs([res, other]); raise AssertionError('different val ids must raise')
        except ValueError as e:
            assert 'fingerprints differ' in str(e)
finally:
    _load_run = _orig_load_run


## 7. The whole flow

`run_eval` evaluates each run, writes one JSON per run (and its per-patch rows, which `held_out` can re-use later), checks the fingerprints, writes `comparison.json`, prints the table, and optionally builds a review report for chosen patch ids. This is what `segpt-eval` calls.

In [ ]:
#| export
def run_eval(runs, thresholds=None, heldout=False, out='eval_out', report_dir=None, report_ids=None,
             report_threshold=0.5, objects_class=None, threshold=0.5, device=None, batch_size=16, echo=print, loader=None):
    "Evaluate finished runs (U-Net and RF-DETR runs side by side; `thresholds=None` = each run's own grid), write `<out>/<run>.json`, `<out>/<run>_per_patch.json` and `<out>/comparison.json`, print one table; returns the comparison dict."
    names = [Path(r).resolve().name for r in runs]
    if len(set(names)) != len(names):
        raise ValueError(f'run directory names must be unique, got {names}')
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    results = []
    for r in runs:
        echo(f'[segpt-eval] {r}')
        res = evaluate_run(r, thresholds, heldout, objects_class, report_ids, report_threshold, device, batch_size, loader)
        per_patch, keep = res.pop('_per_patch'), res.pop('_keep')
        (out / f"{res['run']}.json").write_text(json.dumps(res, indent=1))
        (out / f"{res['run']}_per_patch.json").write_text(json.dumps(per_patch))
        if report_dir is not None and report_ids is not None:
            from segmentation_tools_pt.review_report import Sample, build_report
            missing = [s for s in report_ids if s not in keep]
            if missing:
                raise ValueError(f'{len(missing)} of the {len(report_ids)} report ids are not in the val split of {r}')
            samples = [Sample(s, keep[s][0], keep[s][1].astype(np.uint8), keep[s][2].astype(np.uint8), class_names=res['classes'])
                       for s in sorted(report_ids)]
            echo(f"report: {build_report(res['run'], Path(report_dir), samples)} ({len(samples)} samples)")
        results.append(res)
    cmp = compare_runs(results, threshold)
    (out / 'comparison.json').write_text(json.dumps(cmp, indent=1))
    echo(format_table(cmp))
    return cmp

## Reproducing the #106 held-out table (needs the real cache and runs)

The two U-Net arms of the #106 confirmation (`unet_r18_e15`, `unet_cnx_e15`) must come out of `run_eval(heldout=True)` with the numbers recorded in `docs/superpowers/specs/data/2026-11-23-rfdetr-confirmation-heldout.json`. Machine-specific, so not run by `nbdev-test`.

In [ ]:
#| eval: false
import json
base = Path('/home/hasan-spark/workspace/projects/git_data/segmentation_tools_pt_sanitized/outputs/confirm_106')
runs = [base / f'{a}_e15_s{s}' for a in ('resnet18', 'convnext_tiny') for s in (42, 43)]
cmp = run_eval(runs, heldout=True, out='/tmp/segpt_eval_demo')
ref = json.load(open('docs/superpowers/specs/data/2026-11-23-rfdetr-confirmation-heldout.json'))['arms']
for mine, theirs in (('resnet18_e15', 'unet_r18_e15'), ('convnext_tiny_e15', 'unet_cnx_e15')):
    for k, v in ref[theirs].items():
        assert abs(cmp['arms'][mine][k] - v) < 1e-9, (mine, k, cmp['arms'][mine][k], v)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()